# 10 — Gate AB-4: the ensemble runner (R)

Kernel `R (y2y)`; live internet. Mirrors the parent's `12_gate4_ensemble` + `18_guarded_sweep`
in one pass: for every frozen formulation, at the primary budget level, into
`runs/ab_l/<level>/<formulation_id>/`:

| artifact | what | config |
|---|---|---|
| `anchor/` | engine certified anchor | Gurobi binary, opt_gap 1e-4, NumericFocus |
| `twin/` | LP twin | Gurobi proportion |
| `kbest/` | k-best pool (the two-instrument contrast, E5) | Gurobi binary, portfolio 50 @ 5% |
| `anchor.tif` + `formulation_meta.json` | MGA anchor, drift-checked vs `anchor/` | `mga_core` |
| `mga_g05.tif` | aggregate 5% band, 50 members | `mga_maxham_v1` |
| `mga_guard_g05.tif` | guarded band (per-block floors 0.95) | `mga_block_floors` |
| `mga_g02.tif` + `mga_guard_g02.tif` | **the applied band (D-AB10, g = 2%)**, both semantics — the per-scenario frequency products | same machinery |

Fully resumable per artifact (the reference formulation's AB-1/AB-2 artifacts are reused as-is).
Verifies `manifest.csv` against its freeze hash before solving. At AB scale expect ~2–5 min per
formulation; budget an hour.

In [1]:
# ---- setup + freeze verification ------------------------------------------------------------------------
PROJ <- normalizePath(getwd())
while (!file.exists(file.path(PROJ, "config.py"))) {
  parent <- dirname(PROJ)
  if (identical(parent, PROJ)) stop("config.py not found above getwd() -- open from inside the repo")
  PROJ <- parent
}
setwd(PROJ)
source(file.path(PROJ, "prioritizr_core.R"))
source(file.path(PROJ, "mga_core.R"))
HERE <- file.path(PROJ, "analyses", "alberta_prioritization")
mpath <- file.path(PROJ, "input_data", "aligned_stack_ab", "manifest.json")
stopifnot(file.exists(mpath))
MAN <- read.csv(file.path(HERE, "spec", "manifest.csv"), stringsAsFactors = FALSE)
stopifnot(nrow(MAN) == 14)
dig <- strsplit(readLines(file.path(HERE, "spec", "manifest_freeze.sha256"))[1], "  ")[[1]][1]
stopifnot("manifest.csv does not match its freeze hash -- STOP" =
            identical(unname(tools::sha256sum(file.path(HERE, "spec", "manifest.csv"))[[1]]), dig))
cat(sprintf("manifest verified against freeze hash %s...\n", substr(dig, 1, 16)))
LEVEL <- unique(MAN$budget_level); stopifnot(length(LEVEL) == 1)
BUDGET_PCT <- unique(MAN$budget_pct); stopifnot(length(BUDGET_PCT) == 1)
SC <- jsonlite::read_json(file.path(HERE, "spec", "scenarios_ab_v1.json"))
BLOCKS <- lapply(SC$`_meta`$blocks, unlist)
RUNS_REL <- file.path("analyses/alberta_prioritization/runs/ab_l", LEVEL)
REAL245 <- "input_data/aligned_stack_ab/climate_realizations/macrorefugia_245_2071_2100.tif"
FLOOR_G <- unique(MAN$floor_g); stopifnot(length(FLOOR_G) == 1)
APPLIED_G <- unique(MAN$applied_band_g); stopifnot(length(APPLIED_G) == 1)   # D-AB10: g = 2%
cat(sprintf("level %s | budget_pct %.4f | floors on %s (g %.2f)\n", LEVEL, BUDGET_PCT, paste(names(BLOCKS), collapse = ", "), FLOOR_G))

ctx585 <- pr_setup(mpath, PROJ); ctx585 <- modifyList(ctx585, pr_ingest(ctx585))
ctx245 <- pr_setup(mpath, PROJ)
ctx245$layers$path[ctx245$layers$name == "climate_type_macrorefugia"] <- REAL245
ctx245 <- modifyList(ctx245, pr_ingest(ctx245))
base_for <- function(row) {
  b <- if (grepl("^ssp245", row$climate_level)) ctx245 else ctx585
  b <- pr_override(b, budget_pct = BUDGET_PCT, results_dir = file.path(RUNS_REL, row$formulation_id), results_subdir = "_base")
  modifyList(b, pr_planning_units(b))
}
form_wt <- function(row) list(w = jsonlite::fromJSON(row$weight_vector), t = jsonlite::fromJSON(row$target_vector))

manifest verified against freeze hash 0fc766674b2cb44a...
level A | budget_pct 0.4470 | floors on core_habitat, connectivity, carbon, biodiversity (g 0.05)
prioritizr 8.1.0 | terra 1.9.34 | analysis=ab_y2y | solver=highs (single solution)
objective=min_shortfall | budget=30% target=100% | opt_gap=0.10 | time_limit=43200s
resolution: 1000 m (agg factor 1) | decisions=proportion
roi: mode=full | lock_in=pa_mask
penalties: connectivity=0 | boundary=0 | neighbor=0
outputs -> output_data/ab_y2y
ingested 35 features (8 continuous + 27 EFG) + cost + PA mask | grid 1286 x 3312 @ 1000 m
normalized 35 features to total=100000 each (scale-invariant conditioning)
prioritizr 8.1.0 | terra 1.9.34 | analysis=ab_y2y | solver=highs (single solution)
objective=min_shortfall | budget=30% target=100% | opt_gap=0.10 | time_limit=43200s
resolution: 1000 m (agg factor 1) | decisions=proportion
roi: mode=full | lock_in=pa_mask
penalties: connectivity=0 | boundary=0 | neighbor=0
outputs -> output_data/ab_y2y
i

In [2]:
# ---- DRY PLAN (no solves) ---------------------------------------------------------------------------------
for (i in seq_len(nrow(MAN))) {
  row <- MAN[i, ]; cd <- file.path(PROJ, RUNS_REL, row$formulation_id)
  st <- function(f) if (file.exists(file.path(cd, f))) "done" else "TODO"
  cat(sprintf("%-22s %-7s anchor:%-5s twin:%-5s kbest:%-5s mga05:%-5s guard05:%-5s mga02:%-5s guard02:%-5s\n", row$formulation_id,
              sub("_2071_2100", "", row$climate_level), st("anchor/run_summary.json"), st("twin/run_summary.json"),
              st("kbest/run_summary.json"), st("mga_g05.tif"), st("mga_guard_g05.tif"), st("mga_g02.tif"), st("mga_guard_g02.tif")))
}

s0_ssp585_theta5       ssp585  anchor:done  twin:done  kbest:TODO  mga05:done  guard05:done  mga02:done  guard02:TODO 
s1_ssp585_theta5       ssp585  anchor:TODO  twin:TODO  kbest:TODO  mga05:TODO  guard05:TODO  mga02:TODO  guard02:TODO 
s2_ssp585_theta5       ssp585  anchor:TODO  twin:TODO  kbest:TODO  mga05:TODO  guard05:TODO  mga02:TODO  guard02:TODO 
s3_ssp585_theta5       ssp585  anchor:TODO  twin:TODO  kbest:TODO  mga05:TODO  guard05:TODO  mga02:TODO  guard02:TODO 
s4_ssp585_theta2       ssp585  anchor:done  twin:done  kbest:TODO  mga05:TODO  guard05:TODO  mga02:TODO  guard02:TODO 
s5_ssp585_theta5       ssp585  anchor:TODO  twin:TODO  kbest:TODO  mga05:TODO  guard05:TODO  mga02:TODO  guard02:TODO 
s0_ssp245_theta5       ssp245  anchor:done  twin:done  kbest:TODO  mga05:TODO  guard05:TODO  mga02:TODO  guard02:TODO 
s1_ssp245_theta5       ssp245  anchor:TODO  twin:TODO  kbest:TODO  mga05:TODO  guard05:TODO  mga02:TODO  guard02:TODO 
s2_ssp245_theta5       ssp245  anchor:TODO  twin

In [3]:
# ---- runners -------------------------------------------------------------------------------------------------
run_engine <- function(row, artifact, ov) {
  done <- file.path(PROJ, RUNS_REL, row$formulation_id, artifact, "run_summary.json")
  if (file.exists(done)) { cat(sprintf("   %s/%s exists -- skipped\n", row$formulation_id, artifact)); return(invisible(NULL)) }
  wt <- form_wt(row)
  actx <- do.call(pr_override, c(list(base_for(row), targets = wt$t, feature_weight_multipliers = wt$w,
                                      results_subdir = artifact), ov))
  actx <- modifyList(actx, pr_weights(actx)); actx <- modifyList(actx, pr_targets(actx)); actx <- modifyList(actx, pr_penalty_matrices(actx))
  bp <- pr_build_problem(actx); actx$p <- bp$p; actx$solve_params <- bp$solve_params
  sv <- pr_solve(actx); actx$s <- sv$s; actx$timing <- sv$timing; actx$n_sol <- sv$n_sol; actx$sol_attrs <- sv$sol_attrs
  actx <- modifyList(actx, pr_summaries(actx))
  pr_write_outputs(actx)
  invisible(NULL)
}
run_mga <- function(row) {
  cd <- file.path(PROJ, RUNS_REL, row$formulation_id)
  if (all(file.exists(file.path(cd, c("mga_g05.tif", "mga_guard_g05.tif", "mga_g02.tif", "mga_guard_g02.tif"))))) {
    cat(sprintf("   %s/mga + guard exist -- skipped\n", row$formulation_id)); return(invisible(NULL)) }
  eng <- file.path(cd, "anchor", "run_summary.json"); stopifnot(file.exists(eng))
  z_eng <- as.numeric(unlist(jsonlite::read_json(eng)$solver_provenance$objective))[1]
  wt <- form_wt(row)
  actx <- pr_override(base_for(row), targets = wt$t, feature_weight_multipliers = wt$w, results_subdir = "mga_build",
                      solver = "gurobi", decision_type = "binary", opt_gap = row$opt_gap, portfolio_n = 1)
  actx <- modifyList(actx, pr_weights(actx)); actx <- modifyList(actx, pr_targets(actx)); actx <- modifyList(actx, pr_penalty_matrices(actx))
  bp <- pr_build_problem(actx); actx$p <- bp$p; actx$solve_params <- bp$solve_params
  cm <- mga_compile(actx)
  anchor <- mga_anchor(cm, opt_gap = row$opt_gap)
  rel <- abs(anchor$z - z_eng) / abs(z_eng)
  stopifnot("MGA anchor drifted > 1e-3 from the engine certificate -- STOP" = rel <= 1e-3)
  if (!file.exists(file.path(cd, "anchor.tif"))) {
    r <- terra::rast(actx$cost); v <- rep(NA_integer_, terra::ncell(r)); v[cm$pu_index] <- as.integer(anchor$x)
    terra::values(r) <- v
    terra::writeRaster(r, file.path(cd, "anchor.tif"), overwrite = TRUE, datatype = "INT1U", NAflag = 255,
                       gdal = c("COMPRESS=DEFLATE", "TILED=YES"))
  }
  if (!file.exists(file.path(cd, "formulation_meta.json")))
    jsonlite::write_json(list(formulation_id = row$formulation_id, level = LEVEL, estimator = row$estimator,
                              anchor_objective = anchor$z, anchor_bound = anchor$bound, anchor_gap = anchor$gap,
                              anchor_runtime_s = anchor$runtime, engine_anchor_objective = z_eng, anchor_rel_drift = rel,
                              k = row$k_requested, g = row$band_gap_g, floor_g = FLOOR_G, blocks = BLOCKS,
                              opt_gap = row$opt_gap, mip_gap_dist = 0.01, time_limit_iter = 900,
                              created_utc = format(Sys.time(), tz = "UTC")),
                         file.path(cd, "formulation_meta.json"), auto_unbox = TRUE, pretty = TRUE, digits = 10)
  if (!file.exists(file.path(cd, "mga_g05.tif"))) {
    gen <- mga_generate(cm, anchor, g = row$band_gap_g, k = row$k_requested); mga_write(gen, cm, actx$cost, cd, "g05") }
  if (!file.exists(file.path(cd, "mga_guard_g05.tif"))) {
    gen <- mga_generate(cm, anchor, g = row$band_gap_g, k = row$k_requested,
                        floors = list(ctx = actx, blocks = BLOCKS, g = FLOOR_G))
    mga_write(gen, cm, actx$cost, cd, "guard_g05") }
  # D-AB10: the applied band -- both semantics, every formulation (the per-scenario frequency products)
  if (!file.exists(file.path(cd, "mga_g02.tif"))) {
    gen <- mga_generate(cm, anchor, g = APPLIED_G, k = row$k_requested); mga_write(gen, cm, actx$cost, cd, "g02") }
  if (!file.exists(file.path(cd, "mga_guard_g02.tif"))) {
    gen <- mga_generate(cm, anchor, g = APPLIED_G, k = row$k_requested,
                        floors = list(ctx = actx, blocks = BLOCKS, g = FLOOR_G))
    mga_write(gen, cm, actx$cost, cd, "guard_g02") }
  invisible(NULL)
}

In [4]:
# ---- THE LOOP (serial, resumable anywhere) --------------------------------------------------------------
t_batch <- proc.time()[["elapsed"]]
for (i in seq_len(nrow(MAN))) {
  row <- MAN[i, ]
  cat(sprintf("\n===================== %s (%d/%d) =====================\n", row$formulation_id, i, nrow(MAN)))
  run_engine(row, "anchor", list(solver = "gurobi", decision_type = "binary", opt_gap = row$opt_gap, portfolio_n = 1))
  run_engine(row, "twin",   list(solver = "gurobi", decision_type = "proportion", opt_gap = row$opt_gap, portfolio_n = 1))
  run_engine(row, "kbest",  list(solver = "gurobi", decision_type = "binary", opt_gap = row$opt_gap,
                                 portfolio_n = row$k_requested, portfolio_gap = row$band_gap_g))
  run_mga(row)
  cat(sprintf("== %s done | batch elapsed %.1f min\n", row$formulation_id, (proc.time()[["elapsed"]] - t_batch) / 60))
}
cat("\nAB-4 ENSEMBLE COMPLETE -- next: 11_ab4_analysis.ipynb (kernel y2y-geo)\n")


===================== s0_ssp585_theta5 (1/14) =====================
   s0_ssp585_theta5/anchor exists -- skipped
   s0_ssp585_theta5/twin exists -- skipped
  override budget_pct       -> 0.4470064
  override results_dir      -> analyses/alberta_prioritization/runs/ab_l/A/s0_ssp585_theta5
  override results_subdir   -> _base
  EFFECTIVE targets: <none> | weight multipliers: <none>
  outputs  -> analyses/alberta_prioritization/runs/ab_l/A/s0_ssp585_theta5/_base
planning units: 85,133 cells | budget = 45% = 38,055 cells
locked-in [pa_mask]: 27,972 cells (32.9% of window) -- fits within budget
  override targets          -> irrecoverable_carbon_m_soc=0.322
  override feature_weight_multipliers -> climate_type_macrorefugia=1.038, transboundary_connectivity=0.30626, climate_corridors=1.5978, irrecoverable_carbon_m_soc=0.219191, irrecoverable_carbon_biomass=0.1719, aoh_richness_birds=1.38582, aoh_richness_mammals=2.28103
  override results_subdir   -> kbest
  override solver           -> gur

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (35 total)
│└•planning units:
│ ├•data:       <SpatRaster> (85133 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 38055)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.322 and 1)
││└•weights:    continuous values (between 0.03703704 and 2.281027)
│├•constraints: 
││└•1:          locked in constraints (27972 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   gap portfolio (`number_solutions` = 50, `pool_gap` = 0.05)
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2853457
Set parameter TimeLimit to value 43200
Set parameter NumericFocus to value 2
Set parameter Presolve to value 2
Set parameter Threads to value 10
Set parameter PoolSolutions to value 50
Set parameter PoolSearchMode to value 2
Set parameter PoolGap to value 0.05
WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.6.0 25G83)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  43200
NumericFocus  2
Presolve  2
Threads  10
PoolSolutions  50
PoolSearchMode  2
PoolGap  0.05

WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Optimize a model with 36 rows, 85168 columns and 1194296 nonzeros (Min)
Model fingerprint: 0xdd6917ab
Model has 35 linear objective coefficients
Variable types: 35 conti

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (35 total)
│└•planning units:
│ ├•data:       <SpatRaster> (85133 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 38055)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.322 and 1)
││└•weights:    continuous values (between 0.03703704 and 2.281027)
│├•constraints: 
││└•1:          locked in constraints (27972 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



compiled: 85168 cols (85133 pu + 35 aux) x 36 rows | 27972 locked pu | modelsense min
anchor: objective 4.460914 (bound 4.460914, gap 0.00e+00) | 38,055 selected | 1 s
band wall appended: obj0 . x <= 4.550132  (g = 0.02 on z* = 4.460914)
  block floor core_habitat   anchor capture 0.5568 -> floor 0.5290
  block floor connectivity   anchor capture 0.9928 -> floor 0.9432
  block floor carbon         anchor capture 1.1414 -> floor 1.0843
  block floor biodiversity   anchor capture 0.8624 -> floor 0.8193
g=0.02 iter 01/50: band 4.550128 (+2.00% of z*) OK | ham(anchor) 15,950 | 3 s
g=0.02 iter 02/50: band 4.550131 (+2.00% of z*) OK | ham(anchor) 12,334 | 2 s
g=0.02 iter 03/50: band 4.550123 (+2.00% of z*) OK | ham(anchor) 9,780 | 3 s
g=0.02 iter 04/50: band 4.550122 (+2.00% of z*) OK | ham(anchor) 8,290 | 2 s
g=0.02 iter 05/50: band 4.550130 (+2.00% of z*) OK | ham(anchor) 12,242 | 2 s
g=0.02 iter 06/50: band 4.550132 (+2.00% of z*) OK | ham(anchor) 10,748 | 2 s
g=0.02 iter 07/50: band 4.55

...

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2853457
Set parameter TimeLimit to value 43200
Set parameter NumericFocus to value 2
Set parameter Presolve to value 2
Set parameter Threads to value 10
WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.6.0 25G83)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  43200
NumericFocus  2
Presolve  2
Threads  10

WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Optimize a model with 36 rows, 85168 columns and 1194296 nonzeros (Min)
Model fingerprint: 0x1db9d214
Model has 35 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e-03, 1e+05]
  Objective range  [4e-02, 3e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [4e+04, 1e+05]

Presolve removed 31 rows and 

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (35 total)
│└•planning units:
│ ├•data:       <SpatRaster> (85133 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 38055)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.772 and 1)
││└•weights:    continuous values (between 0.03703704 and 3.342301)
│├•constraints: 
││└•1:          locked in constraints (27972 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   gap portfolio (`number_solutions` = 50, `pool_gap` = 0.05)
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2853457
Set parameter TimeLimit to value 43200
Set parameter NumericFocus to value 2
Set parameter Presolve to value 2
Set parameter Threads to value 10
Set parameter PoolSolutions to value 50
Set parameter PoolSearchMode to value 2
Set parameter PoolGap to value 0.05
WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.6.0 25G83)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  43200
NumericFocus  2
Presolve  2
Threads  10
PoolSolutions  50
PoolSearchMode  2
PoolGap  0.05

WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Optimize a model with 36 rows, 85168 columns and 1194296 nonzeros (Min)
Model fingerprint: 0xcff79343
Model has 35 linear objective coefficients
Variable types: 35 conti

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (35 total)
│└•planning units:
│ ├•data:       <SpatRaster> (85133 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 38055)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.772 and 1)
││└•weights:    continuous values (between 0.03703704 and 3.342301)
│├•constraints: 
││└•1:          locked in constraints (27972 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



compiled: 85168 cols (85133 pu + 35 aux) x 36 rows | 27972 locked pu | modelsense min
anchor: objective 4.632566 (bound 4.632566, gap 0.00e+00) | 38,055 selected | 1 s
band wall appended: obj0 . x <= 4.864194  (g = 0.05 on z* = 4.632566)
g=0.05 iter 01/50: band 4.864193 (+5.00% of z*) OK | ham(anchor) 20,166 | 1 s
g=0.05 iter 02/50: band 4.864192 (+5.00% of z*) OK | ham(anchor) 19,770 | 2 s
g=0.05 iter 03/50: band 4.864170 (+5.00% of z*) OK | ham(anchor) 19,066 | 2 s
g=0.05 iter 04/50: band 4.864157 (+5.00% of z*) OK | ham(anchor) 18,362 | 1 s
g=0.05 iter 05/50: band 4.864148 (+5.00% of z*) OK | ham(anchor) 15,354 | 1 s
g=0.05 iter 06/50: band 4.799674 (+3.61% of z*) OK | ham(anchor) 5,812 | 1 s
g=0.05 iter 07/50: band 4.864186 (+5.00% of z*) OK | ham(anchor) 20,108 | 2 s
g=0.05 iter 08/50: band 4.864167 (+5.00% of z*) OK | ham(anchor) 19,384 | 1 s
g=0.05 iter 09/50: band 4.864132 (+5.00% of z*) OK | ham(anchor) 18,554 | 1 s
g=0.05 iter 10/50: band 4.864168 (+5.00% of z*) OK | ham(anch

In [5]:
# ---- integrity summary ------------------------------------------------------------------------------------
obj_of <- function(p) tryCatch(as.numeric(unlist(jsonlite::read_json(p)$solver_provenance$objective))[1], error = function(e) NA)
for (i in seq_len(nrow(MAN))) {
  row <- MAN[i, ]; cd <- file.path(PROJ, RUNS_REL, row$formulation_id)
  fm <- file.path(cd, "formulation_meta.json"); if (!file.exists(fm)) { cat(sprintf("%-22s INCOMPLETE\n", row$formulation_id)); next }
  m <- jsonlite::read_json(fm); tw <- obj_of(file.path(cd, "twin", "run_summary.json"))
  ce <- read.csv(file.path(cd, "certificates_g05.csv")); cg <- read.csv(file.path(cd, "certificates_guard_g05.csv"))
  c2 <- read.csv(file.path(cd, "certificates_g02.csv")); c2g <- read.csv(file.path(cd, "certificates_guard_g02.csv"))
  kb <- jsonlite::read_json(file.path(cd, "kbest", "run_summary.json"))
  cat(sprintf("%-22s anchor %.6f (%3.0fs, drift %.1e) | twin %.6f [LP<=MILP %s] | kbest %2d | g05 %2d/%s guard %2d/%s | g02 %2d/%s guard %2d/%s | %.1f min\n",
              row$formulation_id, m$anchor_objective, m$anchor_runtime_s, m$anchor_rel_drift, tw,
              ifelse(tw <= m$anchor_objective + 1e-4, "OK", "VIOLATED")   # run_summary objectives are serialized to 4 decimals (jsonlite default) -- M12.1, kb$n_alternatives,
              nrow(ce), if (all(ce$band_ok)) "OK" else "VIOL", nrow(cg), if (all(cg$band_ok)) "OK" else "VIOL",
              nrow(c2), if (all(c2$band_ok)) "OK" else "VIOL", nrow(c2g), if (all(c2g$band_ok)) "OK" else "VIOL",
              (sum(ce$runtime_s) + sum(cg$runtime_s) + sum(c2$runtime_s) + sum(c2g$runtime_s)) / 60))
}

s0_ssp585_theta5       anchor 4.460914 (  1s, drift 3.1e-06) | twin 4.460900 [LP<=MILP OK] | kbest 50 | g05 50/OK guard 50/OK | g02 50/OK guard 50/OK | 5.8 min
s1_ssp585_theta5       anchor 4.331263 (  1s, drift 8.6e-06) | twin 4.331300 [LP<=MILP VIOLATED] | kbest 50 | g05 50/OK guard 50/OK | g02 50/OK guard 50/OK | 7.4 min
s2_ssp585_theta5       anchor 4.450173 (  1s, drift 6.0e-06) | twin 4.450200 [LP<=MILP VIOLATED] | kbest 50 | g05 50/OK guard 50/OK | g02 50/OK guard 50/OK | 6.4 min
s3_ssp585_theta5       anchor 4.629773 (  1s, drift 5.9e-06) | twin 4.629800 [LP<=MILP VIOLATED] | kbest 50 | g05 50/OK guard 50/OK | g02 50/OK guard 50/OK | 6.1 min
s4_ssp585_theta2       anchor 4.288327 (  1s, drift 6.3e-06) | twin 4.288300 [LP<=MILP OK] | kbest 50 | g05 50/OK guard 50/OK | g02 50/OK guard 50/OK | 7.0 min
s5_ssp585_theta5       anchor 9.199058 (  1s, drift 4.6e-06) | twin 9.199100 [LP<=MILP VIOLATED] | kbest 50 | g05 50/OK guard 50/OK | g02 50/OK guard 50/OK | 5.8 min
s0_ssp245_theta5